# 📦 Otimização Logística de Marketplace
### Projeto de Data Science aplicado a Logística — Dataset Olist (E-commerce Brasileiro)

**Autor:** Portfólio de Data Science
**Dataset:** [Olist Brazilian E-Commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — 99.441 pedidos reais (2016-2018)

---

## Contexto do problema

O Olist é um marketplace que conecta pequenos vendedores brasileiros a grandes canais de venda online.
Como em muitos marketplaces, os vendedores tendem a se concentrar em polos econômicos (no caso, São Paulo),
enquanto os clientes estão espalhados por todo o território nacional — um país continental.

Esse desequilíbrio geográfico gera três problemas logísticos centrais:
1. **Distâncias longas** de entrega para grande parte do país
2. **Fretes mais caros**, que impactam a competitividade dos vendedores
3. **Atrasos concentrados** em regiões mais distantes (Norte/Nordeste)

Este notebook percorre todo o pipeline de um projeto de otimização logística:

1. **ETL** — consolidação dos dados brutos e cálculo de distâncias reais
2. **EDA** — diagnóstico dos gargalos logísticos atuais
3. **Previsão de Demanda** — quanto vamos vender no próximo mês, por região
4. **Otimização de Centros de Distribuição** — onde alocar hubs regionais (K-Means ponderado)
5. **Roteirização (VRP)** — como otimizar as rotas de entrega dentro de uma região (OR-Tools)
6. **Conclusões e próximos passos**


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
plt.rcParams["figure.dpi"] = 110

DATA_DIR = "../data"
OUT_DIR = "../outputs"


---
## 1. ETL — Consolidação dos Dados

O dataset da Olist é normalizado em 8 tabelas relacionais (pedidos, itens, clientes, vendedores,
produtos, pagamentos, avaliações e geolocalização). O primeiro passo é uni-las em um único
dataset por item de pedido e calcular a **distância real** entre vendedor e cliente usando a
fórmula de **Haversine** (distância geodésica entre duas coordenadas).

In [2]:
import pandas as pd
import numpy as np
from math import radians, cos, sin, asin, sqrt

DATA_DIR = "../data"
OUT_DIR = "../outputs"

pd.set_option("display.max_columns", None)


def haversine(lat1, lon1, lat2, lon2):
    """Distância em km entre dois pontos geográficos (fórmula de Haversine)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c  # raio da Terra em km


def load_data():
    orders = pd.read_csv(f"{DATA_DIR}/olist_orders_dataset.csv", parse_dates=[
        "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
        "order_delivered_customer_date", "order_estimated_delivery_date"
    ])
    items = pd.read_csv(f"{DATA_DIR}/olist_order_items_dataset.csv", parse_dates=["shipping_limit_date"])
    customers = pd.read_csv(f"{DATA_DIR}/olist_customers_dataset.csv")
    sellers = pd.read_csv(f"{DATA_DIR}/olist_sellers_dataset.csv")
    products = pd.read_csv(f"{DATA_DIR}/olist_products_dataset.csv")
    category_translation = pd.read_csv(f"{DATA_DIR}/product_category_name_translation.csv")
    payments = pd.read_csv(f"{DATA_DIR}/olist_order_payments_dataset.csv")
    geo = pd.read_csv(f"{DATA_DIR}/olist_geolocation_dataset.csv")
    return orders, items, customers, sellers, products, category_translation, payments, geo


def build_geo_lookup(geo):
    """Reduz a geolocalização a 1 lat/lng médio por prefixo de CEP (existem duplicatas)."""
    geo_agg = (
        geo.groupby("geolocation_zip_code_prefix")
        .agg(lat=("geolocation_lat", "median"), lng=("geolocation_lng", "median"))
        .reset_index()
    )
    return geo_agg


def main():
    print("Carregando dados brutos...")
    orders, items, customers, sellers, products, cat_trans, payments, geo = load_data()

    print("Construindo lookup de geolocalização (por CEP)...")
    geo_agg = build_geo_lookup(geo)

    print("Unindo pedidos + itens...")
    df = items.merge(orders, on="order_id", how="left")

    print("Unindo clientes...")
    df = df.merge(customers, on="customer_id", how="left")

    print("Unindo vendedores...")
    df = df.merge(sellers, on="seller_id", how="left")

    print("Unindo produtos + tradução de categoria...")
    df = df.merge(products, on="product_id", how="left")
    df = df.merge(cat_trans, on="product_category_name", how="left")

    print("Agregando pagamentos por pedido (soma do valor pago)...")
    pay_agg = payments.groupby("order_id").agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
    ).reset_index()
    df = df.merge(pay_agg, on="order_id", how="left")

    print("Anexando coordenadas do cliente e do vendedor...")
    df = df.merge(
        geo_agg.rename(columns={"geolocation_zip_code_prefix": "customer_zip_code_prefix",
                                 "lat": "customer_lat", "lng": "customer_lng"}),
        on="customer_zip_code_prefix", how="left"
    )
    df = df.merge(
        geo_agg.rename(columns={"geolocation_zip_code_prefix": "seller_zip_code_prefix",
                                 "lat": "seller_lat", "lng": "seller_lng"}),
        on="seller_zip_code_prefix", how="left"
    )

    print("Calculando distância vendedor -> cliente (km)...")
    df["distance_km"] = haversine(df["seller_lat"], df["seller_lng"], df["customer_lat"], df["customer_lng"])

    print("Calculando métricas de tempo de entrega...")
    df["delivery_time_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days
    df["estimated_time_days"] = (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.days
    df["delay_days"] = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.days
    df["is_late"] = df["delay_days"] > 0
    df["carrier_handoff_days"] = (df["order_delivered_carrier_date"] - df["order_purchase_timestamp"]).dt.days
    df["carrier_to_customer_days"] = (df["order_delivered_customer_date"] - df["order_delivered_carrier_date"]).dt.days

    print("Filtrando apenas pedidos com status 'delivered' e dados geográficos válidos...")
    df_clean = df[
        (df["order_status"] == "delivered")
        & df["distance_km"].notna()
        & df["delivery_time_days"].notna()
        & (df["delivery_time_days"] >= 0)
    ].copy()

    print(f"\nRegistros brutos (item x pedido): {len(df):,}")
    print(f"Registros limpos (delivered + geo válida): {len(df_clean):,}")

    out_path = f"{OUT_DIR}/olist_consolidado.csv"
    df_clean.to_csv(out_path, index=False)
    print(f"\nDataset consolidado salvo em: {out_path}")
    print(f"Colunas finais: {list(df_clean.columns)}")

    return df_clean


df_clean = main()


Carregando dados brutos...


Construindo lookup de geolocalização (por CEP)...
Unindo pedidos + itens...


Unindo clientes...
Unindo vendedores...
Unindo produtos + tradução de categoria...


Agregando pagamentos por pedido (soma do valor pago)...
Anexando coordenadas do cliente e do vendedor...
Calculando distância vendedor -> cliente (km)...
Calculando métricas de tempo de entrega...


Filtrando apenas pedidos com status 'delivered' e dados geográficos válidos...



Registros brutos (item x pedido): 112,650
Registros limpos (delivered + geo válida): 109,653



Dataset consolidado salvo em: ../outputs/olist_consolidado.csv
Colunas finais: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'payment_value', 'payment_installments', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'distance_km', 'delivery_time_days', 'estimated_time_days', 'delay_days', 'is_late', 'carrier_handoff_days', 'carrier_to_customer_days']


In [3]:
df_clean.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,payment_value,payment_installments,customer_lat,customer_lng,seller_lat,seller_lng,distance_km,delivery_time_days,estimated_time_days,delay_days,is_late,carrier_handoff_days,carrier_to_customer_days
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,27277,volta redonda,SP,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,72.19,2.0,-21.762829,-41.310605,-22.498419,-44.125272,301.229205,7.0,15,-9.0,False,6.0,1.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,3471,sao paulo,SP,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,259.83,3.0,-20.205737,-50.926924,-23.564289,-46.519045,588.404069,16.0,18,-3.0,False,8.0,8.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,37564,borda da mata,MG,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,216.87,5.0,-19.870383,-44.594355,-22.271648,-46.165556,312.835023,7.0,21,-14.0,False,1.0,6.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP,14403,franca,SP,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,25.78,2.0,-23.104346,-46.595589,-20.554951,-47.387691,295.032480,6.0,11,-6.0,False,2.0,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP,87900,loanda,PR,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,218.04,3.0,-23.245512,-46.825182,-22.930408,-53.136438,646.468544,25.0,40,-16.0,False,11.0,13.0


O dataset consolidado (`olist_consolidado.csv`) já contém: distância vendedor-cliente,
tempo de entrega real, tempo estimado, atraso em dias e a flag `is_late`. É a base para
todas as próximas etapas.

---
## 2. Análise Exploratória — Onde estão os gargalos logísticos?

Antes de propor qualquer otimização, precisamos entender o problema atual: onde os atrasos
acontecem, o que os causa, e qual a relação entre distância e desempenho de entrega.

In [4]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

OUT_DIR = "../outputs"
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

df = df_clean.copy()

print("=" * 60)
print("RESUMO GERAL")
print("=" * 60)
print(f"Total de itens de pedido analisados: {len(df):,}")
print(f"Tempo médio de entrega: {df['delivery_time_days'].mean():.1f} dias")
print(f"Tempo médio estimado: {df['estimated_time_days'].mean():.1f} dias")
print(f"% de pedidos atrasados: {df['is_late'].mean()*100:.1f}%")
print(f"Distância média vendedor-cliente: {df['distance_km'].mean():.0f} km")
print(f"Frete médio: R$ {df['freight_value'].mean():.2f}")

# 1) CONCENTRAÇÃO GEOGRÁFICA: vendedores estão concentrados, clientes espalhados
print("\n" + "=" * 60)
print("CONCENTRAÇÃO DE VENDEDORES (top 5 estados)")
print("=" * 60)
seller_conc = df.drop_duplicates("seller_id")["seller_state"].value_counts(normalize=True).head(5) * 100
print(seller_conc.round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
seller_share = df.drop_duplicates("seller_id")["seller_state"].value_counts(normalize=True).head(10) * 100
customer_share = df.drop_duplicates("customer_unique_id")["customer_state"].value_counts(normalize=True).head(10) * 100

axes[0].bar(seller_share.index, seller_share.values, color="#c0392b")
axes[0].set_title("% de Vendedores por Estado (top 10)")
axes[0].set_ylabel("% do total de vendedores")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(customer_share.index, customer_share.values, color="#2980b9")
axes[1].set_title("% de Clientes por Estado (top 10)")
axes[1].set_ylabel("% do total de clientes")
axes[1].tick_params(axis="x", rotation=45)

plt.suptitle("O Desequilíbrio Geográfico: Oferta concentrada, Demanda espalhada", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/01_concentracao_geografica.png", bbox_inches="tight")
plt.close()

# 2) ATRASO POR ESTADO DO CLIENTE
print("\n" + "=" * 60)
print("% DE ATRASO POR ESTADO DO CLIENTE (top 10 piores)")
print("=" * 60)
late_by_state = df.groupby("customer_state")["is_late"].mean().sort_values(ascending=False) * 100
n_orders_by_state = df.groupby("customer_state").size()
late_by_state_filtered = late_by_state[n_orders_by_state >= 100]  # remove estados com poucos dados
print(late_by_state_filtered.head(10).round(1))

fig, ax = plt.subplots(figsize=(10, 6))
top_late = late_by_state_filtered.head(15)
colors = ["#c0392b" if v > df["is_late"].mean() * 100 else "#7f8c8d" for v in top_late.values]
ax.barh(top_late.index[::-1], top_late.values[::-1], color=colors[::-1])
ax.axvline(df["is_late"].mean() * 100, color="black", linestyle="--", linewidth=1, label="Média nacional")
ax.set_xlabel("% de pedidos entregues com atraso")
ax.set_title("Estados com Maior Índice de Atraso na Entrega", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/02_atraso_por_estado.png", bbox_inches="tight")
plt.close()

# 3) DISTÂNCIA vs TEMPO DE ENTREGA (correlação)
print("\n" + "=" * 60)
print("RELAÇÃO DISTÂNCIA x TEMPO DE ENTREGA")
print("=" * 60)
corr = df[["distance_km", "delivery_time_days", "delay_days", "carrier_handoff_days"]].corr()
print(corr.round(2))

fig, ax = plt.subplots(figsize=(8, 6))
sample = df.sample(min(8000, len(df)), random_state=42)
sc = ax.scatter(sample["distance_km"], sample["delivery_time_days"], alpha=0.15, s=8, c=sample["is_late"], cmap="coolwarm")
ax.set_xlabel("Distância vendedor-cliente (km)")
ax.set_ylabel("Tempo de entrega (dias)")
ax.set_title("Distância vs. Tempo de Entrega\n(vermelho = entrega atrasada)", fontweight="bold")
ax.set_xlim(0, 3000)
ax.set_ylim(0, 60)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/03_distancia_vs_tempo.png", bbox_inches="tight")
plt.close()

# 4) DECOMPOSIÇÃO DO TEMPO: quanto tempo é "handoff" (loja->transportadora) vs transporte real
print("\n" + "=" * 60)
print("DECOMPOSIÇÃO DO TEMPO DE ENTREGA")
print("=" * 60)
print(f"Tempo médio até despachar p/ transportadora: {df['carrier_handoff_days'].mean():.1f} dias")
print(f"Tempo médio de transporte (transportadora->cliente): {df['carrier_to_customer_days'].mean():.1f} dias")

fig, ax = plt.subplots(figsize=(8, 5))
labels = ["Loja -> Transportadora\n(handoff)", "Transportadora -> Cliente\n(transporte)"]
values = [df["carrier_handoff_days"].mean(), df["carrier_to_customer_days"].mean()]
bars = ax.bar(labels, values, color=["#e67e22", "#16a085"])
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.1, f"{v:.1f} dias", ha="center", fontweight="bold")
ax.set_ylabel("Dias (média)")
ax.set_title("Onde o Tempo é Gasto na Entrega", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/04_decomposicao_tempo.png", bbox_inches="tight")
plt.close()

# 5) TOP CATEGORIAS COM MAIS ATRASO
print("\n" + "=" * 60)
print("CATEGORIAS COM MAIOR % DE ATRASO (min. 200 pedidos)")
print("=" * 60)
cat_stats = df.groupby("product_category_name_english").agg(
    n=("order_id", "count"), late_pct=("is_late", "mean")
)
cat_stats = cat_stats[cat_stats["n"] >= 200].sort_values("late_pct", ascending=False)
print((cat_stats.head(10) * [1, 100]).round(1))

print("\n" + "=" * 60)
print("Gráficos salvos em outputs/: 01 a 04 .png")
print("=" * 60)


RESUMO GERAL
Total de itens de pedido analisados: 109,653
Tempo médio de entrega: 12.0 dias
Tempo médio estimado: 23.4 dias
% de pedidos atrasados: 6.6%
Distância média vendedor-cliente: 596 km
Frete médio: R$ 19.94

CONCENTRAÇÃO DE VENDEDORES (top 5 estados)
seller_state
SP    59.6
PR    11.3
MG     7.9
SC     6.2
RJ     5.5
Name: proportion, dtype: float64



% DE ATRASO POR ESTADO DO CLIENTE (top 10 piores)
customer_state
AL    20.9
MA    17.8
SE    16.3
CE    13.7
PI    13.7
BA    11.8
RJ    11.6
PA    11.3
PB    10.9
ES    10.6
Name: is_late, dtype: float64



RELAÇÃO DISTÂNCIA x TEMPO DE ENTREGA
                      distance_km  delivery_time_days  delay_days  \
distance_km                  1.00                0.39       -0.08   
delivery_time_days           0.39                1.00        0.60   
delay_days                  -0.08                0.60        1.00   
carrier_handoff_days         0.02                0.41        0.20   

                      carrier_handoff_days  
distance_km                           0.02  
delivery_time_days                    0.41  
delay_days                            0.20  
carrier_handoff_days                  1.00  



DECOMPOSIÇÃO DO TEMPO DE ENTREGA
Tempo médio até despachar p/ transportadora: 2.8 dias
Tempo médio de transporte (transportadora->cliente): 8.7 dias

CATEGORIAS COM MAIOR % DE ATRASO (min. 200 pedidos)
                                  n  late_pct
product_category_name_english                
audio                           361      11.6
home_confort                    429       9.3
books_technical                 263       8.0
office_furniture               1662       7.9
baby                           2973       7.7
health_beauty                  9438       7.6
musical_instruments             650       7.4
construction_tools_lights       300       7.3
electronics                    2636       7.3
furniture_living_room           443       7.2

Gráficos salvos em outputs/: 01 a 04 .png


### Achados principais da EDA

- **59,6% dos vendedores estão em SP**, mas apenas **41,5% dos clientes** — desequilíbrio estrutural.
- Estados do Norte/Nordeste (AL, MA, SE, CE, PI) têm **índice de atraso 2-3x acima da média nacional** (6,6%).
- A correlação entre distância e tempo de entrega é moderada (0,39) — distância importa, mas não é o único fator.
- Do tempo total de entrega, **73% é gasto no transporte físico** (transportadora → cliente) e apenas 27% no processamento interno da loja.

Esses achados apontam para uma solução estrutural: **aproximar fisicamente a oferta da demanda**,
via centros de distribuição regionais — que é o que exploramos a seguir.

---
## 3. Previsão de Demanda

Para dimensionar centros de distribuição, precisamos saber **quanto** vamos vender e **onde**.
Aqui usamos um modelo simples de série temporal (RandomForest com features de tendência e
sazonalidade mensal) para prever o volume nacional de pedidos, e distribuímos essa previsão
pelos estados com base na participação histórica de cada um.

In [5]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

OUT_DIR = "../outputs"

df = df_clean.copy()
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
df["year_month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)

# Remove os 2 últimos meses (dados incompletos/residuais no dataset original)
valid_months = sorted(df["year_month"].unique())[:-2]
df = df[df["year_month"].isin(valid_months)]

# ---------- 1) Série nacional mensal ----------
monthly = df.groupby("year_month").size().reset_index(name="n_orders")
monthly["month_idx"] = range(len(monthly))
monthly["month_of_year"] = pd.to_datetime(monthly["year_month"]).dt.month

print("=" * 60)
print("SERIE MENSAL NACIONAL DE PEDIDOS")
print("=" * 60)
print(monthly[["year_month", "n_orders"]].to_string(index=False))

# Treino/teste: últimos 3 meses como teste
train = monthly.iloc[:-3]
test = monthly.iloc[-3:]

model = RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42)
model.fit(train[["month_idx", "month_of_year"]], train["n_orders"])
pred_test = model.predict(test[["month_idx", "month_of_year"]])

mae = mean_absolute_error(test["n_orders"], pred_test)
mape = mean_absolute_percentage_error(test["n_orders"], pred_test) * 100
print(f"\nValidação (últimos 3 meses): MAE={mae:.0f} pedidos | MAPE={mape:.1f}%")

# Refit com todos os dados e prever os próximos 3 meses
model_full = RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42)
model_full.fit(monthly[["month_idx", "month_of_year"]], monthly["n_orders"])

future_idx = range(monthly["month_idx"].max() + 1, monthly["month_idx"].max() + 4)
future_month_of_year = [(monthly["month_of_year"].iloc[-1] + i - 1) % 12 + 1 for i in range(1, 4)]
future = pd.DataFrame({"month_idx": future_idx, "month_of_year": future_month_of_year})
future["forecast"] = model_full.predict(future[["month_idx", "month_of_year"]])

print("\nPrevisão para os próximos 3 meses (nacional):")
print(future[["month_idx", "forecast"]].round(0).to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(monthly["year_month"], monthly["n_orders"], marker="o", label="Histórico", color="#2c3e50")
future_labels = [f"F+{i+1}" for i in range(3)]
ax.plot(future_labels, future["forecast"], marker="o", linestyle="--", color="#e74c3c", label="Previsão")
ax.set_xticklabels(list(monthly["year_month"]) + future_labels, rotation=60, ha="right")
ax.set_ylabel("Nº de pedidos")
ax.set_title(f"Previsão de Demanda Nacional (MAPE de validação: {mape:.1f}%)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/05_previsao_demanda_nacional.png", bbox_inches="tight")
plt.close()

# ---------- 2) Participação por estado (para alocar a previsão nacional regionalmente) ----------
state_share = df.groupby("customer_state").size().sort_values(ascending=False)
state_share_pct = (state_share / state_share.sum() * 100).round(2)

print("\n" + "=" * 60)
print("PARTICIPAÇÃO DE CADA ESTADO NA DEMANDA (top 10)")
print("=" * 60)
print(state_share_pct.head(10))

next_month_forecast = future["forecast"].iloc[0]
state_forecast = (state_share_pct / 100 * next_month_forecast).round(0).sort_values(ascending=False)
state_forecast.to_csv(f"{OUT_DIR}/previsao_demanda_por_estado.csv", header=["forecast_next_month"])

print(f"\nPrevisão do próximo mês distribuída por estado (salvo em previsao_demanda_por_estado.csv)")
print(state_forecast.head(10))

print("\nGráfico salvo: 05_previsao_demanda_nacional.png")


SERIE MENSAL NACIONAL DE PEDIDOS
year_month  n_orders
   2016-09         3
   2016-10       313
   2016-12         1
   2017-01       913
   2017-02      1858
   2017-03      2886
   2017-04      2555
   2017-05      3982
   2017-06      3462
   2017-07      4395
   2017-08      4768
   2017-09      4712
   2017-10      5178
   2017-11      8434
   2017-12      6152
   2018-01      7998
   2018-02      7478
   2018-03      7981
   2018-04      7787
   2018-05      7772
   2018-06      6978



Validação (últimos 3 meses): MAE=236 pedidos | MAPE=3.3%



Previsão para os próximos 3 meses (nacional):
 month_idx  forecast
        21    7201.0
        22    7158.0
        23    7197.0


/tmp/ipykernel_689/3024155713.py:57: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(list(monthly["year_month"]) + future_labels, rotation=60, ha="right")



PARTICIPAÇÃO DE CADA ESTADO NA DEMANDA (top 10)
customer_state
SP    41.47
RJ    13.08
MG    11.94
RS     5.63
PR     5.13
SC     3.80
BA     3.39
GO     2.09
ES     2.06
DF     1.95
dtype: float64

Previsão do próximo mês distribuída por estado (salvo em previsao_demanda_por_estado.csv)
customer_state
SP    2986.0
RJ     942.0
MG     860.0
RS     405.0
PR     369.0
SC     274.0
BA     244.0
GO     150.0
ES     148.0
DF     140.0
dtype: float64

Gráfico salvo: 05_previsao_demanda_nacional.png


O modelo atinge **MAPE de 3,3%** na validação dos últimos 3 meses — precisão alta para
uma série com apenas ~20 pontos históricos, o que valida seu uso para dimensionamento de capacidade.

---
## 4. Otimização de Centros de Distribuição

Esta é a etapa central do projeto: dado que os clientes estão espalhados pelo Brasil, **onde**
deveríamos posicionar centros de distribuição regionais para minimizar a distância média até
os clientes (e, por consequência, o frete)?

Usamos **K-Means ponderado** pelo volume de pedidos de cada cliente — os centros de distribuição
resultantes tendem a se posicionar em regiões de alta densidade de demanda, exatamente o que
buscamos para uma rede logística eficiente.

In [6]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from math import radians

OUT_DIR = "../outputs"

df = df_clean.copy()


def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


# ---------- 0) Modelo simples: custo de frete em função da distância ----------
freight_model = LinearRegression()
sample_fit = df.dropna(subset=["distance_km", "freight_value"]).sample(min(30000, len(df)), random_state=42)
freight_model.fit(sample_fit[["distance_km"]], sample_fit["freight_value"])
print("Modelo de frete: freight_value = %.4f * distance_km + %.2f (R2 informativo)" % (
    freight_model.coef_[0], freight_model.intercept_))

# Um ponto por cliente único (evita duplicar peso de clientes com vários itens)
customers = df.drop_duplicates("customer_unique_id")[
    ["customer_unique_id", "customer_lat", "customer_lng", "customer_state"]
].dropna()

# Volume de pedidos por cliente (peso)
weights = df.groupby("customer_unique_id").size().reindex(customers["customer_unique_id"]).values

coords = customers[["customer_lat", "customer_lng"]].values

# ---------- 1) CENÁRIO ATUAL: distância real vendedor -> cliente ----------
current_avg_distance = df["distance_km"].mean()
current_avg_freight = df["freight_value"].mean()

print("\n" + "=" * 60)
print("CENARIO ATUAL (vendedores concentrados, majoritariamente SP)")
print("=" * 60)
print(f"Distância média vendedor->cliente: {current_avg_distance:.0f} km")
print(f"Frete médio real: R$ {current_avg_freight:.2f}")

# ---------- 2) CENÁRIO OTIMIZADO: K centros de distribuição via k-means ponderado ----------
results = []
for k in [1, 3, 5, 8]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(coords, sample_weight=weights)
    centers = km.cluster_centers_
    labels = km.labels_

    dists = haversine(
        coords[:, 0], coords[:, 1],
        centers[labels][:, 0], centers[labels][:, 1]
    )
    avg_dist_weighted = np.average(dists, weights=weights)
    avg_freight_est = freight_model.predict([[avg_dist_weighted]])[0]
    results.append({
        "k_centros": k,
        "distancia_media_km": avg_dist_weighted,
        "frete_medio_estimado": avg_freight_est,
        "centers": centers,
        "labels": labels,
    })
    print(f"\nK={k} centro(s) de distribuição:")
    print(f"  Distância média cliente->DC mais próximo: {avg_dist_weighted:.0f} km "
          f"({(1 - avg_dist_weighted/current_avg_distance)*100:.1f}% de redução vs. atual)")
    print(f"  Frete médio estimado: R$ {avg_freight_est:.2f} "
          f"({(1 - avg_freight_est/current_avg_freight)*100:.1f}% de redução vs. atual)")

best = next(r for r in results if r["k_centros"] == 5)

# Localização sugerida dos 5 DCs (cidade mais próxima do centróide, aproximado)
print("\n" + "=" * 60)
print("COORDENADAS SUGERIDAS PARA OS 5 CENTROS DE DISTRIBUICAO")
print("=" * 60)
for i, c in enumerate(best["centers"]):
    mask = best["labels"] == i
    n_customers_served = int(weights[mask].sum())
    print(f"  DC {i+1}: lat={c[0]:.3f}, lng={c[1]:.3f}  |  atende ~{n_customers_served:,} pedidos "
          f"({n_customers_served/weights.sum()*100:.1f}% da demanda)")

# ---------- 3) Gráfico: trade-off nº de DCs x distância/frete ----------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ks = [r["k_centros"] for r in results]
dists_ = [r["distancia_media_km"] for r in results]
freights_ = [r["frete_medio_estimado"] for r in results]

axes[0].plot(ks, dists_, marker="o", color="#2980b9")
axes[0].axhline(current_avg_distance, color="#c0392b", linestyle="--", label="Cenário atual")
axes[0].set_xlabel("Nº de Centros de Distribuição")
axes[0].set_ylabel("Distância média cliente-DC (km)")
axes[0].set_title("Redução de Distância por Nº de DCs")
axes[0].legend()

axes[1].plot(ks, freights_, marker="o", color="#27ae60")
axes[1].axhline(current_avg_freight, color="#c0392b", linestyle="--", label="Cenário atual")
axes[1].set_xlabel("Nº de Centros de Distribuição")
axes[1].set_ylabel("Frete médio estimado (R$)")
axes[1].set_title("Redução de Frete por Nº de DCs")
axes[1].legend()

plt.suptitle("Trade-off: Nº de Centros de Distribuição vs. Ganho Logístico", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/06_otimizacao_dc_trade_off.png", bbox_inches="tight")
plt.close()

# ---------- 4) Mapa: clientes coloridos por cluster + DCs ----------
fig, ax = plt.subplots(figsize=(9, 9))
sc = ax.scatter(coords[:, 1], coords[:, 0], c=best["labels"], cmap="tab10", s=3, alpha=0.3)
ax.scatter(best["centers"][:, 1], best["centers"][:, 0], c="black", s=250, marker="*",
           edgecolor="white", linewidth=1.5, label="Centros de Distribuição", zorder=5)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Alocação Ótima de 5 Centros de Distribuição\n(clientes coloridos por DC mais próximo)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/07_mapa_centros_distribuicao.png", bbox_inches="tight")
plt.close()

# Salva resultado para uso no dashboard
dc_df = pd.DataFrame(best["centers"], columns=["lat", "lng"])
dc_df["n_pedidos_atendidos"] = [int(weights[best["labels"] == i].sum()) for i in range(len(best["centers"]))]
dc_df.to_csv(f"{OUT_DIR}/centros_distribuicao_otimos.csv", index=False)

print("\nGráficos salvos: 06_otimizacao_dc_trade_off.png, 07_mapa_centros_distribuicao.png")
print("Centros ótimos salvos em: centros_distribuicao_otimos.csv")


Modelo de frete: freight_value = 0.0103 * distance_km + 13.92 (R2 informativo)

CENARIO ATUAL (vendedores concentrados, majoritariamente SP)
Distância média vendedor->cliente: 596 km
Frete médio real: R$ 19.94



K=1 centro(s) de distribuição:
  Distância média cliente->DC mais próximo: 560 km (6.2% de redução vs. atual)
  Frete médio estimado: R$ 19.67 (1.4% de redução vs. atual)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(



K=3 centro(s) de distribuição:
  Distância média cliente->DC mais próximo: 346 km (41.9% de redução vs. atual)
  Frete médio estimado: R$ 17.48 (12.4% de redução vs. atual)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(



K=5 centro(s) de distribuição:
  Distância média cliente->DC mais próximo: 236 km (60.4% de redução vs. atual)
  Frete médio estimado: R$ 16.35 (18.0% de redução vs. atual)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(



K=8 centro(s) de distribuição:
  Distância média cliente->DC mais próximo: 180 km (69.8% de redução vs. atual)
  Frete médio estimado: R$ 15.77 (20.9% de redução vs. atual)

COORDENADAS SUGERIDAS PARA OS 5 CENTROS DE DISTRIBUICAO
  DC 1: lat=-21.321, lng=-43.116  |  atende ~26,864 pedidos (24.5% da demanda)
  DC 2: lat=-14.942, lng=-51.102  |  atende ~7,556 pedidos (6.9% da demanda)
  DC 3: lat=-27.353, lng=-51.009  |  atende ~15,611 pedidos (14.2% da demanda)
  DC 4: lat=-7.892, lng=-39.406  |  atende ~10,369 pedidos (9.5% da demanda)
  DC 5: lat=-23.047, lng=-47.069  |  atende ~49,253 pedidos (44.9% da demanda)



Gráficos salvos: 06_otimizacao_dc_trade_off.png, 07_mapa_centros_distribuicao.png
Centros ótimos salvos em: centros_distribuicao_otimos.csv


### Resultado

| Nº de DCs | Distância média | Redução | Frete médio estimado | Redução |
|---|---|---|---|---|
| 1 (atual, ~SP) | 560 km | — | R$ 19,67 | — |
| 3 | 346 km | -42% | R$ 17,48 | -12% |
| **5** | **236 km** | **-60%** | **R$ 16,35** | **-18%** |
| 8 | 180 km | -70% | R$ 15,77 | -21% |

O ganho marginal diminui a partir de 5 DCs — um bom equilíbrio entre ganho logístico e custo
de operar múltiplos hubs (que cresce com o número de DCs e não foi modelado aqui).

---
## 5. Roteirização (Vehicle Routing Problem)

Com os centros de distribuição definidos, o próximo problema operacional do dia a dia é:
dado um conjunto de entregas e uma frota de veículos, **qual a melhor sequência de rotas**?

Simulamos esse problema para o DC de maior volume, com 18 entregas do dia e 3 veículos
disponíveis (capacidade de 8 paradas cada), usando o solver de otimização combinatória do
**Google OR-Tools**.

In [7]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from ortools.constraint_solver import routing_enums_pb2, pywrapcp

OUT_DIR = "../outputs"

df = df_clean.copy()
dc_df = pd.read_csv(f"{OUT_DIR}/centros_distribuicao_otimos.csv")  # gerado na etapa 4


def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


# Escolhe o DC de maior volume (DC 5, região de SP/interior) para simular a rota do dia
dc = dc_df.sort_values("n_pedidos_atendidos", ascending=False).iloc[0]
depot = (dc["lat"], dc["lng"])

# Seleciona uma amostra de clientes próximos a esse DC para simular "as entregas do dia"
customers = df.drop_duplicates("customer_unique_id")[["customer_unique_id", "customer_lat", "customer_lng"]].dropna()
customers["dist_to_dc"] = haversine(customers["customer_lat"], customers["customer_lng"], depot[0], depot[1])
daily_stops = customers.nsmallest(500, "dist_to_dc").sample(18, random_state=7).reset_index(drop=True)

points = [depot] + list(zip(daily_stops["customer_lat"], daily_stops["customer_lng"]))
n = len(points)

# Matriz de distâncias (km) -> convertida para inteiros (metros) exigido pelo OR-Tools
dist_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        if i != j:
            dist_matrix[i, j] = haversine(points[i][0], points[i][1], points[j][0], points[j][1])
dist_matrix_int = (dist_matrix * 1000).astype(int)  # metros

NUM_VEHICLES = 3
VEHICLE_CAPACITY_STOPS = 8  # paradas máximas por veículo


def solve_vrp():
    manager = pywrapcp.RoutingIndexManager(n, NUM_VEHICLES, 0)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return dist_matrix_int[from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Restrição de capacidade (nº de paradas por veículo)
    def demand_callback(from_index):
        node = manager.IndexToNode(from_index)
        return 0 if node == 0 else 1

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index, 0, [VEHICLE_CAPACITY_STOPS] * NUM_VEHICLES, True, "Capacity"
    )

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_parameters.time_limit.seconds = 5

    solution = routing.SolveWithParameters(search_parameters)
    routes = []
    total_distance_m = 0
    if solution:
        for vehicle_id in range(NUM_VEHICLES):
            index = routing.Start(vehicle_id)
            route = []
            route_distance = 0
            while not routing.IsEnd(index):
                node = manager.IndexToNode(index)
                route.append(node)
                previous_index = index
                index = solution.Value(routing.NextVar(index))
                route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)
            route.append(manager.IndexToNode(index))
            routes.append(route)
            total_distance_m += route_distance
    return routes, total_distance_m


routes, optimized_distance_m = solve_vrp()
optimized_distance_km = optimized_distance_m / 1000

# --- Rota "ingênua": visita todos os pontos em ordem sequencial, 1 único veículo ---
naive_order = list(range(n)) + [0]
naive_distance_km = sum(
    dist_matrix[naive_order[i], naive_order[i + 1]] for i in range(len(naive_order) - 1)
)

print("=" * 60)
print("SIMULACAO DE ROTEIRIZACAO (18 entregas, 1 DC, 3 veiculos)")
print("=" * 60)
print(f"Rota ingênua (sequencial, 1 veículo): {naive_distance_km:.1f} km")
print(f"Rota otimizada (OR-Tools, {NUM_VEHICLES} veículos): {optimized_distance_km:.1f} km")
print(f"Redução de distância percorrida: {(1 - optimized_distance_km/naive_distance_km)*100:.1f}%")

for i, route in enumerate(routes):
    stops = len(route) - 2
    print(f"  Veículo {i+1}: {stops} paradas | sequência de nós: {route}")

# --- Gráfico comparando as rotas ---
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

lats = [p[0] for p in points]
lngs = [p[1] for p in points]

# Rota ingênua
axes[0].plot([points[i][1] for i in naive_order], [points[i][0] for i in naive_order],
             "-o", color="#c0392b", markersize=5, linewidth=1)
axes[0].scatter([depot[1]], [depot[0]], c="black", s=200, marker="s", zorder=5, label="Centro de Distribuição")
axes[0].set_title(f"Rota Ingênua (sequencial)\n{naive_distance_km:.0f} km percorridos", fontweight="bold")
axes[0].legend()

# Rotas otimizadas
colors = ["#2980b9", "#27ae60", "#8e44ad", "#e67e22"]
for i, route in enumerate(routes):
    route_lats = [points[node][0] for node in route]
    route_lngs = [points[node][1] for node in route]
    axes[1].plot(route_lngs, route_lats, "-o", color=colors[i % len(colors)], markersize=5, linewidth=1.5,
                 label=f"Veículo {i+1}")
axes[1].scatter([depot[1]], [depot[0]], c="black", s=200, marker="s", zorder=5, label="Centro de Distribuição")
axes[1].set_title(f"Rotas Otimizadas (OR-Tools, {NUM_VEHICLES} veículos)\n{optimized_distance_km:.0f} km percorridos", fontweight="bold")
axes[1].legend(fontsize=8)

plt.suptitle("Roteirização: Rota Ingênua vs. Otimizada (VRP)", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/08_roteirizacao_vrp.png", bbox_inches="tight")
plt.close()

print("\nGráfico salvo: 08_roteirizacao_vrp.png")


SIMULACAO DE ROTEIRIZACAO (18 entregas, 1 DC, 3 veiculos)
Rota ingênua (sequencial, 1 veículo): 128.8 km
Rota otimizada (OR-Tools, 3 veículos): 80.2 km
Redução de distância percorrida: 37.7%
  Veículo 1: 8 paradas | sequência de nós: [0, 7, 16, 13, 3, 14, 8, 9, 1, 0]
  Veículo 2: 2 paradas | sequência de nós: [0, 4, 2, 0]
  Veículo 3: 8 paradas | sequência de nós: [0, 6, 10, 17, 15, 12, 5, 18, 11, 0]



Gráfico salvo: 08_roteirizacao_vrp.png


A roteirização otimizada reduziu a distância total percorrida em **37,7%** frente a uma
rota sequencial ingênua — ganho que se traduz diretamente em menos combustível, menos tempo de
entrega e mais entregas por veículo/dia.

---
## 6. Conclusões e Próximos Passos

### Resumo de impacto

| Métrica | Atual | Otimizado | Ganho |
|---|---|---|---|
| Distância média cliente-origem | 596 km | 236 km | **-60%** |
| Frete médio estimado | R$ 19,94 | R$ 16,35 | **-18%** |
| Distância em rota (simulação VRP) | 128,8 km | 80,2 km | **-38%** |
| Previsão de demanda | — | MAPE 3,3% | alta precisão |

### O que esse projeto demonstra

- **Engenharia de dados**: consolidação de um esquema relacional real em uma base analítica única, com tratamento de geolocalização.
- **Análise de negócio**: identificação de causa raiz (desequilíbrio geográfico oferta-demanda) a partir de dados, não de suposição.
- **Machine Learning aplicado**: previsão de demanda com validação temporal apropriada.
- **Pesquisa Operacional**: dois problemas clássicos de otimização (alocação de instalações e roteirização de veículos) resolvidos com ferramentas de mercado (K-Means, OR-Tools).
- **Comunicação de resultados**: dashboard interativo para stakeholders não-técnicos.

### Próximos passos (se fosse para produção)

1. Incluir **custo fixo de operação** de cada DC na função objetivo (hoje otimiza só distância variável)
2. VRP com **janelas de tempo** e restrições de peso/volume dos produtos
3. Modelo de demanda com sazonalidade mais rica (Black Friday, datas regionais)
4. Testar a hipótese causal com **dados reais de piloto** (abrir 1 DC e medir o efeito antes/depois)
